# Week 3 Assignment – Customer Intelligence System

## Classification, Ensemble Learning & Clustering on Country Data

**Author:** Sahil Yadav  
**Internship:** Celebal Technologies – Data Science  
**Dataset:** [Unsupervised Learning on Country Data](https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data)  
**Reference:** [Kaggle Notebook](https://www.kaggle.com/code/jatin2bagga/unsupervised-learning-on-country-data/notebook)  

---

### Objective
Develop an end-to-end **Customer Intelligence System** using classification, ensemble learning (Random Forest, XGBoost), and clustering (K-Means, DBSCAN), achieving optimized predictive performance and actionable customer segmentation insights.

### Approach
1. Load, clean and explore the Country Data
2. Engineer a target variable (development category) for classification
3. Train **Random Forest** and **XGBoost** classifiers (ensemble learning)
4. Evaluate with Accuracy, Precision, Recall, F1, Confusion Matrix
5. Extract **Feature Importance**
6. Apply **K-Means** and **DBSCAN** clustering for unsupervised segmentation
7. Profile clusters and derive actionable insights

---
## 1. Install & Import Libraries

In [1]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

# Classification / Ensemble
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    silhouette_score, davies_bouldin_score
)

# Clustering
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

# Dendrogram (optional viz)
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 100)

print('All libraries imported successfully!')

---
## 2. Load & Inspect the Dataset

In [ ]:
import io, os

try:
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(iter(uploaded))
    df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
except ImportError:
    possible = [
        'Country-data.csv',
        '../Country-data.csv',
        'data/Country-data.csv',
        '/kaggle/input/unsupervised-learning-on-country-data/Country-data.csv'
    ]
    for p in possible:
        if os.path.exists(p):
            df = pd.read_csv(p)
            print(f'Loaded from: {p}')
            break
    else:
        raise FileNotFoundError("Place 'Country-data.csv' in the working directory.")

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()
print('\n')
df.describe().T

In [ ]:
print('Missing values:\n', df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

---
## 3. Data Cleaning & Preprocessing

In [ ]:
df_clean = df.copy()
df_clean.columns = [c.strip().lower() for c in df_clean.columns]
df_clean = df_clean.drop_duplicates()

numeric_cols = [c for c in df_clean.columns if c != 'country']
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())

print('Missing values after cleaning:', df_clean.isnull().sum().sum())
df_clean.head()

---
## 4. Exploratory Data Analysis

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(df_clean[numeric_cols].corr(), dtype=bool))
sns.heatmap(df_clean[numeric_cols].corr(), mask=mask, annot=True,
            cmap='coolwarm', fmt='.2f', linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key features
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for idx, col in enumerate(numeric_cols):
    ax = axes[idx // 3, idx % 3]
    sns.histplot(df_clean[col], kde=True, ax=ax, color='steelblue', bins=25)
    ax.set_title(col, fontsize=12, fontweight='bold')
plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Target Variable Engineering

Since the dataset is unlabeled, we create a **development category** target using multiple socio-economic indicators for classification.

In [ ]:
def categorize_country(row):
    """Categorize countries: 0=Underdeveloped, 1=Developing, 2=Developed."""
    score = 0
    if row['income'] > 20000:  score += 2
    elif row['income'] > 5000: score += 1
    if row['child_mort'] < 10:   score += 2
    elif row['child_mort'] < 40: score += 1
    if row['life_expec'] > 75:   score += 2
    elif row['life_expec'] > 65: score += 1
    if row['gdpp'] > 15000:  score += 2
    elif row['gdpp'] > 3000: score += 1
    
    if score >= 6: return 2
    elif score >= 3: return 1
    else: return 0

df_clean['dev_category'] = df_clean.apply(categorize_country, axis=1)
label_map = {0: 'Underdeveloped', 1: 'Developing', 2: 'Developed'}

print('Class Distribution:')
print(df_clean['dev_category'].map(label_map).value_counts())

plt.figure(figsize=(8, 5))
df_clean['dev_category'].map(label_map).value_counts().plot(
    kind='bar', color=['#e74c3c', '#f39c12', '#27ae60'])
plt.title('Development Category Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 6. Feature Scaling & Train-Test Split

In [ ]:
X = df_clean[numeric_cols].values
y = df_clean['dev_category'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Train classes: {np.bincount(y_train)}')
print(f'Test  classes: {np.bincount(y_test)}')

---
# PART A — Classification & Ensemble Learning

---
## 7. Random Forest Classifier

Random Forest is a **bagging** ensemble of decision trees. Each tree trains on a bootstrap sample with random feature subsets, reducing overfitting.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_split=5,
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_acc  = accuracy_score(y_test, rf_pred)
rf_prec = precision_score(y_test, rf_pred, average='weighted', zero_division=0)
rf_rec  = recall_score(y_test, rf_pred, average='weighted', zero_division=0)
rf_f1   = f1_score(y_test, rf_pred, average='weighted', zero_division=0)
rf_cv   = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')

print('=' * 55)
print(' RANDOM FOREST — Results')
print('=' * 55)
print(f' Accuracy  : {rf_acc:.4f}')
print(f' Precision : {rf_prec:.4f}')
print(f' Recall    : {rf_rec:.4f}')
print(f' F1-Score  : {rf_f1:.4f}')
print(f' CV Mean   : {rf_cv.mean():.4f} (+/- {rf_cv.std():.4f})')
print('\nClassification Report:')
print(classification_report(y_test, rf_pred, target_names=list(label_map.values())))

In [ ]:
# Confusion Matrix
cm_rf = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(label_map.values()),
            yticklabels=list(label_map.values()))
plt.title('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## 8. Feature Importance (Random Forest)

In [ ]:
feat_imp = pd.DataFrame({
    'Feature': numeric_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(feat_imp)))
plt.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors)
plt.xlabel('Importance')
plt.title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(feat_imp.sort_values('Importance', ascending=False).to_string(index=False))

---
## 9. XGBoost Classifier

XGBoost (Extreme Gradient Boosting) is a **boosting** ensemble that builds trees sequentially, each correcting the errors of the previous one. It includes built-in L1/L2 regularization.

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, use_label_encoder=False, eval_metric='mlogloss',
    verbosity=0
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_acc  = accuracy_score(y_test, xgb_pred)
xgb_prec = precision_score(y_test, xgb_pred, average='weighted', zero_division=0)
xgb_rec  = recall_score(y_test, xgb_pred, average='weighted', zero_division=0)
xgb_f1   = f1_score(y_test, xgb_pred, average='weighted', zero_division=0)
xgb_cv   = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='accuracy')

print('=' * 55)
print(' XGBOOST — Results')
print('=' * 55)
print(f' Accuracy  : {xgb_acc:.4f}')
print(f' Precision : {xgb_prec:.4f}')
print(f' Recall    : {xgb_rec:.4f}')
print(f' F1-Score  : {xgb_f1:.4f}')
print(f' CV Mean   : {xgb_cv.mean():.4f} (+/- {xgb_cv.std():.4f})')
print('\nClassification Report:')
print(classification_report(y_test, xgb_pred, target_names=list(label_map.values())))

In [ ]:
# Confusion Matrix
cm_xgb = confusion_matrix(y_test, xgb_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=list(label_map.values()),
            yticklabels=list(label_map.values()))
plt.title('Confusion Matrix — XGBoost', fontsize=13, fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## 10. Classification Model Comparison

In [ ]:
comp = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost'],
    'Accuracy': [rf_acc, xgb_acc],
    'Precision': [rf_prec, xgb_prec],
    'Recall': [rf_rec, xgb_rec],
    'F1-Score': [rf_f1, xgb_f1],
    'CV Mean': [rf_cv.mean(), xgb_cv.mean()]
})
print('\nModel Comparison:')
display(comp)

# Bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'CV Mean']
x = np.arange(len(metrics))
w = 0.3

plt.figure(figsize=(10, 5))
plt.bar(x - w/2, comp.iloc[0][metrics], w, label='Random Forest', color='#3498db')
plt.bar(x + w/2, comp.iloc[1][metrics], w, label='XGBoost', color='#e67e22')
plt.xticks(x, metrics)
plt.ylabel('Score')
plt.ylim(0, 1.15)
plt.title('Random Forest vs XGBoost', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
# PART B — Clustering (Unsupervised Segmentation)

We now apply clustering **without** the target variable to discover natural country groupings.

---
## 11. K-Means Clustering

K-Means partitions data into K clusters by iteratively assigning points to the nearest centroid and updating centroids.

In [ ]:
X_cluster = scaler.fit_transform(df_clean[numeric_cols])

# Elbow method + Silhouette to find optimal K
k_range = range(2, 11)
inertias, sil_scores = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertias, 'bo-', lw=2)
ax1.set_title('Elbow Method', fontsize=13, fontweight='bold')
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia')

ax2.plot(k_range, sil_scores, 'ro-', lw=2)
best_k = list(k_range)[np.argmax(sil_scores)]
ax2.axvline(x=best_k, color='green', ls='--', label=f'Best K={best_k}')
ax2.set_title('Silhouette Score', fontsize=13, fontweight='bold')
ax2.set_xlabel('K'); ax2.set_ylabel('Score'); ax2.legend()

plt.tight_layout()
plt.show()
print(f'Optimal K (silhouette) = {best_k}')

In [ ]:
# Fit K-Means with K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clean['kmeans_cluster'] = kmeans.fit_predict(X_cluster)

km_sil = silhouette_score(X_cluster, df_clean['kmeans_cluster'])
km_db  = davies_bouldin_score(X_cluster, df_clean['kmeans_cluster'])

print(f'K-Means (K=3)')
print(f'  Silhouette Score    : {km_sil:.4f}')
print(f'  Davies-Bouldin Index: {km_db:.4f}')
print(f'\nCluster sizes:\n{df_clean["kmeans_cluster"].value_counts().sort_index()}')

In [ ]:
# PCA visualisation
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df_clean['kmeans_cluster'],
                      cmap='Set1', alpha=0.7, edgecolors='k', linewidth=0.5, s=80)
centroids_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='black', marker='X',
            s=200, edgecolors='yellow', linewidths=2, label='Centroids')
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('K-Means Clusters (PCA)', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## 12. DBSCAN Clustering

DBSCAN (Density-Based Spatial Clustering) groups densely packed points and marks sparse-region points as **noise (−1)**. It does not require specifying K upfront.

In [ ]:
# K-distance graph to choose eps
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_cluster)
distances, _ = nn.kneighbors(X_cluster)
distances = np.sort(distances[:, -1])

plt.figure(figsize=(10, 5))
plt.plot(distances, lw=2, color='#8e44ad')
plt.title('K-Distance Graph (eps selection)', fontsize=14, fontweight='bold')
plt.xlabel('Points (sorted)'); plt.ylabel('5-NN Distance')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
dbscan = DBSCAN(eps=2.0, min_samples=5)
df_clean['dbscan_cluster'] = dbscan.fit_predict(X_cluster)

n_clusters_db = len(set(df_clean['dbscan_cluster'])) - (1 if -1 in df_clean['dbscan_cluster'].values else 0)
n_noise = (df_clean['dbscan_cluster'] == -1).sum()

print(f'DBSCAN Results:')
print(f'  Clusters found: {n_clusters_db}')
print(f'  Noise points  : {n_noise}')
print(f'\n{df_clean["dbscan_cluster"].value_counts().sort_index()}')

non_noise = df_clean['dbscan_cluster'] != -1
if non_noise.sum() > 0 and len(df_clean[non_noise]['dbscan_cluster'].unique()) > 1:
    db_sil = silhouette_score(X_cluster[non_noise], df_clean.loc[non_noise, 'dbscan_cluster'])
    db_dbi = davies_bouldin_score(X_cluster[non_noise], df_clean.loc[non_noise, 'dbscan_cluster'])
    print(f'\n  Silhouette (excl. noise): {db_sil:.4f}')
    print(f'  Davies-Bouldin (excl. noise): {db_dbi:.4f}')

In [ ]:
# PCA visualisation
plt.figure(figsize=(10, 7))
unique_labels = sorted(df_clean['dbscan_cluster'].unique())
colors = plt.cm.Set1(np.linspace(0, 1, len(unique_labels)))

for lbl, clr in zip(unique_labels, colors):
    mask = df_clean['dbscan_cluster'] == lbl
    if lbl == -1:
        plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c='gray', marker='x',
                    s=50, alpha=0.5, label='Noise')
    else:
        plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[clr], alpha=0.7,
                    edgecolors='k', linewidth=0.5, s=80, label=f'Cluster {lbl}')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('DBSCAN Clusters (PCA)', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## 13. Clustering Comparison

In [ ]:
# Side-by-side PCA
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=df_clean['kmeans_cluster'],
            cmap='Set1', alpha=0.7, edgecolors='k', linewidth=0.3, s=60)
ax1.set_title('K-Means', fontsize=13, fontweight='bold')
ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2')

ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=df_clean['dbscan_cluster'],
            cmap='Set2', alpha=0.7, edgecolors='k', linewidth=0.3, s=60)
ax2.set_title('DBSCAN', fontsize=13, fontweight='bold')
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')

plt.suptitle('K-Means vs DBSCAN — Cluster Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 14. Cluster Profiling & Customer Segmentation Insights

In [ ]:
print('=' * 60)
print(' K-MEANS CLUSTER PROFILES')
print('=' * 60)

profile = df_clean.groupby('kmeans_cluster')[numeric_cols].mean().round(2)
display(profile)

for cid in sorted(df_clean['kmeans_cluster'].unique()):
    grp = df_clean[df_clean['kmeans_cluster'] == cid]
    print(f'\n--- Cluster {cid} ({len(grp)} countries) ---')
    print(f'  Avg Income: ${grp["income"].mean():,.0f}  |  Avg GDPP: ${grp["gdpp"].mean():,.0f}')
    print(f'  Avg Child Mort: {grp["child_mort"].mean():.1f}  |  Avg Life Exp: {grp["life_expec"].mean():.1f}')
    if 'country' in grp.columns:
        names = grp['country'].tolist()
        print(f'  Countries: {", ".join(names[:10])}{"..." if len(names)>10 else ""}')

In [ ]:
# Countries needing immediate aid
cluster_means = df_clean.groupby('kmeans_cluster')[['child_mort','income','gdpp']].mean()
vulnerable = cluster_means['child_mort'].idxmax()
aid = df_clean[df_clean['kmeans_cluster'] == vulnerable].sort_values('child_mort', ascending=False)

print('\n' + '=' * 60)
print(' COUNTRIES NEEDING IMMEDIATE AID')
print('=' * 60)
if 'country' in aid.columns:
    display(aid[['country','child_mort','income','life_expec','gdpp']].head(15))

---
## 15. Final Summary

In [ ]:
print('=' * 65)
print(' FINAL SUMMARY')
print('=' * 65)
print('\n CLASSIFICATION (Ensemble Learning):')
print(f'   Random Forest — Accuracy: {rf_acc:.4f}  F1: {rf_f1:.4f}')
print(f'   XGBoost       — Accuracy: {xgb_acc:.4f}  F1: {xgb_f1:.4f}')
best_cls = 'Random Forest' if rf_acc >= xgb_acc else 'XGBoost'
print(f'   Best: {best_cls}')

print('\n CLUSTERING (Customer Segmentation):')
print(f'   K-Means — Silhouette: {km_sil:.4f}  DB Index: {km_db:.4f}')
if non_noise.sum() > 0 and len(df_clean[non_noise]['dbscan_cluster'].unique()) > 1:
    print(f'   DBSCAN  — Silhouette: {db_sil:.4f}  DB Index: {db_dbi:.4f}')

print('\n KEY INSIGHTS:')
print('  1. Countries form 3 natural segments: Developed, Developing, Underdeveloped.')
print('  2. child_mort, income, gdpp, life_expec are the most discriminating features.')
print('  3. Ensemble models (RF, XGBoost) achieve strong classification accuracy.')
print('  4. K-Means produces clean, interpretable segments for aid prioritisation.')
print('  5. DBSCAN identifies outlier economies that do not fit typical clusters.')
print('\n' + '=' * 65)
print(' Notebook Complete — Sahil Yadav | Celebal Technologies')
print('=' * 65)